# 21 · Gray-Box AIA72 Historical Real-Outage Replay

Phase E. Uses **actual AIA acquisition/recovery logs** from the upstream `solar-flare-aia-training` project to replay detected AIA unavailability against the frozen 72-hour Gray-Box predictions.

This is post-hoc robustness analysis only. It does **not** claim historical operational delivery, does not infer silent corruption, and does not tune the fallback policy. It distinguishes observed acquisition incidents from synthetic random outages.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, re
import numpy as np
import pandas as pd

ROOT=Path('/home/abmoses2000')
SEEDS=[17,29,43]
PRIMARY_ALPHA=.10
PRIMARY_PCT=.90

MASTER=ROOT/'graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz'
MASTER_SHA='17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46'
SHARP=ROOT/'sharp72_predictions_frozen_20261002.csv.gz'
SHARP_SHA='7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b'
CASES=ROOT/'graybox_aia72_inference_v1_20261004/inputs/cases.csv.gz'
MOND=ROOT/'graybox_aia72_conformal_mondrian_v2_20261005/mondrian_thresholds.csv'
CUTS=ROOT/'graybox_aia72_policy_freeze_v1_20261005/candidate_cutoffs.csv'
ALARM=ROOT/'graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005/alarm_thresholds.csv'

OUT=ROOT/'graybox_aia72_real_outage_replay_v1_20261006'
if OUT.exists():
    raise SystemExit(f'Refusing to overwrite existing output: {OUT}')
OUT.mkdir(parents=True)

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for c in iter(lambda:f.read(4*1024*1024),b''):
            h.update(c)
    return h.hexdigest()

def require(c,m):
    if not c:
        raise ValueError(m)

for p in [MASTER,SHARP,CASES,MOND,CUTS,ALARM]:
    require(p.is_file(),f'Missing frozen Gray-Box input: {p}')
require(sha256(MASTER)==MASTER_SHA,'Master SHA changed')
require(sha256(SHARP)==SHARP_SHA,'SHARP SHA changed')

# ------------------------------------------------------------
# Locate upstream AIA acquisition metadata read-only.
# Prefer a local clone; otherwise fetch exact files from a pinned upstream Git commit.
# ------------------------------------------------------------
import urllib.request

UPSTREAM_COMMIT='9dd35d301e153030261ea78d5419ad5e7bf5eda4'
UPSTREAM_REPO='Watchman77/solar-flare-aia-training'

repo_candidates=[
    ROOT/'solar_flare_aia',
    ROOT/'solar-flare-aia-training',
    ROOT/'solar_flare_aia_training',
    ROOT/'solar_flare_aia'/'solar-flare-aia-training',
]
AIA_REPO=None
for p in repo_candidates:
    if (p/'harp_block_miner').is_dir():
        AIA_REPO=p
        break

cache=OUT/'upstream_aia_metadata'
cache.mkdir(parents=True,exist_ok=True)

wanted=[
    ('2025','harp_block_miner/production_aia2025/metadata/recovery_inventory_aia2025.csv'),
    ('2025','harp_block_miner/production_aia2025/metadata/block_log_aia2025.csv'),
    ('2025','harp_block_miner/production_aia2025/metadata/sample_log_aia2025.csv'),
    ('2026','harp_block_miner/production_aia2026/metadata/recovery_inventory_aia2026.csv'),
    ('2026','harp_block_miner/production_aia2026/metadata/block_log_aia2026.csv'),
    ('2026','harp_block_miner/production_aia2026/metadata/sample_log_aia2026.csv'),
    ('2026_recovery','harp_block_miner/recovery_aia2026-recovery/metadata/recovery_audit_aia2026-recovery.csv'),
    ('2026_recovery','harp_block_miner/recovery_aia2026-recovery/metadata/recovery_group_log_aia2026-recovery.csv'),
    ('2026_recovery','harp_block_miner/recovery_aia2026-recovery/metadata/recovery_plan_aia2026-recovery.csv'),
    ('2026_recovery','harp_block_miner/recovery_aia2026-recovery/metadata/remaining_recovery_ids_aia2026-recovery.txt'),
]

sources={}
for group,rel in wanted:
    local=(AIA_REPO/rel) if AIA_REPO is not None else None
    if local is not None and local.is_file():
        p=local
    else:
        p=cache/Path(rel).name
        url=f'https://raw.githubusercontent.com/{UPSTREAM_REPO}/{UPSTREAM_COMMIT}/{rel}'
        try:
            urllib.request.urlretrieve(url,p)
        except Exception as e:
            if p.exists():
                p.unlink()
            print('Upstream source unavailable:',rel,repr(e))
            continue
    sources[f'{group}:{Path(rel).name}']=p

require(any('recovery_inventory' in k for k in sources),'No real recovery inventory found from local clone or pinned GitHub commit')

source_manifest=[]
for k,p in sources.items():
    source_manifest.append({
        'source_key':k,'path':str(p),'bytes':p.stat().st_size,'sha256':sha256(p),
        'upstream_repo':UPSTREAM_REPO,'upstream_commit':UPSTREAM_COMMIT,
        'retrieval':'local_clone' if AIA_REPO is not None and str(p).startswith(str(AIA_REPO)) else 'pinned_raw_github'
    })
pd.DataFrame(source_manifest).to_csv(OUT/'upstream_source_manifest.csv',index=False)

# ------------------------------------------------------------
# Load frozen Gray-Box corpus and reconstruct frozen v3 q90
# ------------------------------------------------------------
a=pd.read_csv(MASTER,low_memory=False)
s=pd.read_csv(SHARP,low_memory=False)
cases=pd.read_csv(CASES,low_memory=False)
for d in [a,s,cases]:
    d['forecast_case_id']=d['forecast_case_id'].astype(str)
m=a.merge(s,on='forecast_case_id',suffixes=('_aia','_sharp'),validate='one_to_one')
keep=['forecast_case_id']
for c in ['source_sample_id','HARPNUM','NOAA_AR_clean','role','issue_utc']:
    if c in cases.columns: keep.append(c)
m=m.merge(cases[keep],on='forecast_case_id',how='left',suffixes=('','_case'),validate='one_to_one')
require(len(m)==len(a),'Alignment loss')

la='label_aia' if 'label_aia' in m.columns else 'label'
ls='label_sharp' if 'label_sharp' in m.columns else la
m['label_final']=pd.to_numeric(m[la],errors='raise').astype(int)
require(np.array_equal(m.label_final,pd.to_numeric(m[ls],errors='raise').astype(int)),'Label mismatch')

role_col='role_aia' if 'role_aia' in m.columns else ('role_sharp' if 'role_sharp' in m.columns else 'role')
m['role_final']=m[role_col].astype(str)

# issue time
issue=pd.Series(pd.NaT,index=m.index,dtype='datetime64[ns, UTC]')
for c in [x for x in ['issue_utc_aia','issue_utc_sharp','issue_utc_case','issue_utc'] if x in m.columns]:
    z=pd.to_datetime(m[c],utc=True,errors='coerce'); take=issue.isna()&z.notna(); issue.loc[take]=z.loc[take]
id_time=m.forecast_case_id.str.extract(r':(\d{8}_\d{4})_',expand=False)
z=pd.to_datetime(id_time,format='%Y%m%d_%H%M',utc=True,errors='coerce'); take=issue.isna()&z.notna(); issue.loc[take]=z.loc[take]
require(issue.notna().all(),'Unresolved issue time')
m['issue_dt']=issue
m['year']=issue.dt.year.astype(int)

m['sharp_probability']=pd.to_numeric(m['probability_gru_mean'],errors='coerce')
m['aia_probability']=pd.to_numeric(m['probability'],errors='coerce')
m['fusion_probability']=.5*(m.sharp_probability+m.aia_probability)

def seed_cols(side):
    out=[]
    for z in SEEDS:
        c=f'probability_seed_{z}_{side}'
        if c in m.columns: out.append(c)
        elif f'probability_seed_{z}' in m.columns: out.append(f'probability_seed_{z}')
        else: raise KeyError(f'Missing seed {z} for {side}')
    return out

AS=m[seed_cols('aia')].astype(float).to_numpy()
SS=m[seed_cols('sharp')].astype(float).to_numpy()
FS=.5*(AS+SS)
m['fusion_seed_std']=np.nanstd(FS,axis=1,ddof=1)
m['sharp_aia_probability_gap']=abs(m.sharp_probability-m.aia_probability)
q=np.clip(m.fusion_probability.to_numpy(float),1e-12,1-1e-12)
m['fusion_entropy']=-(q*np.log(q)+(1-q)*np.log(1-q))

alarm=pd.read_csv(ALARM)
ath={r.branch:float(r.threshold) for r in alarm.itertuples()}
m['sharp_alarm']=(m.sharp_probability>=ath['sharp']).astype(int)
m['fusion_alarm']=(m.fusion_probability>=ath['sharp_aia']).astype(int)

mond=pd.read_csv(MOND)
z=mond[(mond.branch=='sharp_aia')&np.isclose(mond.alpha.astype(float),PRIMARY_ALPHA)]
require(len(z)==1,'Missing fusion Mondrian q90')
q0,q1=float(z.iloc[0].qhat_class0),float(z.iloc[0].qhat_class1)
p=m.fusion_probability.to_numpy(float)
inc0=p<=q0; inc1=(1-p)<=q1
fusion_single=(inc0&~inc1)|(~inc0&inc1)

cuts=pd.read_csv(CUTS)
def cutoff(metric):
    z=cuts[(cuts.metric==metric)&np.isclose(cuts.percentile.astype(float),PRIMARY_PCT)]
    require(len(z)==1,f'Missing q90 cutoff {metric}')
    return float(z.iloc[0].cutoff)

aia_status='input_status' if 'input_status' in m.columns else ('input_status_aia' if 'input_status_aia' in m.columns else None)
aia_ok=np.isfinite(m.aia_probability)
if aia_status is not None:
    aia_ok &= m[aia_status].astype(str).eq('ok').to_numpy()
sharp_ok=np.isfinite(m.sharp_probability)
fusion_ok=aia_ok&sharp_ok&np.isfinite(m.fusion_probability)
normal=(fusion_ok&fusion_single&
        (m.fusion_seed_std.to_numpy(float)<=cutoff('fusion_seed_std'))&
        (m.sharp_aia_probability_gap.to_numpy(float)<=cutoff('sharp_aia_probability_gap'))&
        (m.fusion_entropy.to_numpy(float)<=cutoff('fusion_entropy')))
m['v3_base_state']=np.where(normal,'NORMAL',np.where(sharp_ok,'DEGRADED','ABSTAIN'))
m['v3_base_prediction']=np.where(normal,m.fusion_alarm,np.where(sharp_ok,m.sharp_alarm,np.nan))

# ------------------------------------------------------------
# Build observed real-outage incident inventory
# ------------------------------------------------------------
inventories=[]
for year in [2025,2026]:
    key=f'{year}:recovery_inventory_aia{year}.csv'
    if key not in sources: continue
    x=pd.read_csv(sources[key],low_memory=False)
    x['year']=year
    x['T_REC_dt']=pd.to_datetime(x['T_REC_dt'],utc=True,errors='coerce')
    inventories.append(x)
inc=pd.concat(inventories,ignore_index=True) if inventories else pd.DataFrame()
require(len(inc)>0,'Empty recovery inventory')
require({'sample_id','T_REC_dt','HARPNUM','recovery_class','block_id'}.issubset(inc.columns),'Recovery inventory schema changed')
inc.to_csv(OUT/'real_acquisition_incidents.csv',index=False)

incident_summary=inc.groupby(['year','recovery_class']).agg(
    incident_samples=('sample_id','size'),
    unique_blocks=('block_id','nunique'),
    unique_harps=('HARPNUM','nunique'),
    first_time=('T_REC_dt','min'),
    last_time=('T_REC_dt','max')
).reset_index()
incident_summary.to_csv(OUT/'real_incident_summary.csv',index=False)

block_summary=inc.groupby(['year','block_id','recovery_class']).agg(
    samples=('sample_id','size'),first_time=('T_REC_dt','min'),last_time=('T_REC_dt','max'),HARPNUM=('HARPNUM','first')
).reset_index()
block_summary['span_hours']=(block_summary.last_time-block_summary.first_time).dt.total_seconds()/3600
block_summary.to_csv(OUT/'real_incident_blocks.csv',index=False)

# Persistent 2026 missing IDs after recovery, if retained.
remaining_ids=set()
rk='2026_recovery:remaining_recovery_ids_aia2026-recovery.txt'
if rk in sources:
    remaining_ids={x.strip() for x in sources[rk].read_text().splitlines() if x.strip()}
inc['persistent_after_recovery']=inc['sample_id'].astype(str).isin(remaining_ids)
inc[['sample_id','year','recovery_class','block_id','persistent_after_recovery']].to_csv(
    OUT/'recovery_persistence.csv',index=False)

# ------------------------------------------------------------
# Map real incidents to Gray-Box cases
# Prefer exact source_sample_id; fallback to time+HARP when necessary.
# ------------------------------------------------------------
m['_map_sample_id']=m['source_sample_id'].astype(str) if 'source_sample_id' in m.columns else ''
exact_map={sid:i for i,sid in enumerate(m['_map_sample_id']) if sid and sid!='nan'}
mapped=[]
for r in inc.itertuples():
    idx=exact_map.get(str(r.sample_id))
    method='source_sample_id' if idx is not None else None
    if idx is None and 'HARPNUM' in m.columns:
        harp=pd.to_numeric(m.HARPNUM,errors='coerce')
        same=(harp==float(r.HARPNUM)) & (m.issue_dt==r.T_REC_dt)
        hits=np.flatnonzero(same.to_numpy())
        if len(hits)==1:
            idx=int(hits[0]); method='harp_time'
    if idx is not None:
        mapped.append({'sample_id':r.sample_id,'row_index':idx,'mapping_method':method,
                       'recovery_class':r.recovery_class,'block_id':r.block_id,'year':r.year,
                       'persistent_after_recovery':bool(getattr(r,'persistent_after_recovery',False))})
mapped=pd.DataFrame(mapped)
mapped.to_csv(OUT/'incident_to_graybox_mapping.csv',index=False)

# ------------------------------------------------------------
# Replay real detected AIA outages on matching 2025/2026 cases.
# Two scenarios:
#   incident_replay = every acquisition incident is treated AIA-unavailable
#   persistent_replay = only IDs still missing after recovery
# ------------------------------------------------------------
def binary_metrics(y,pred):
    y=np.asarray(y,int); pred=np.asarray(pred,int)
    tp=int(((y==1)&(pred==1)).sum()); fp=int(((y==0)&(pred==1)).sum())
    fn=int(((y==1)&(pred==0)).sum()); tn=int(((y==0)&(pred==0)).sum())
    rec=tp/(tp+fn) if tp+fn else np.nan; spec=tn/(tn+fp) if tn+fp else np.nan
    prec=tp/(tp+fp) if tp+fp else np.nan
    f1=2*prec*rec/(prec+rec) if np.isfinite(prec) and np.isfinite(rec) and prec+rec else np.nan
    return {'tp':tp,'fp':fp,'fn':fn,'tn':tn,'recall':rec,'specificity':spec,'precision':prec,
            'f1':f1,'tss':rec+spec-1 if np.isfinite(rec) and np.isfinite(spec) else np.nan}

def replay(mask,scenario):
    rows=[]
    for role in ['retrospective_cycle25','supplementary_2026']:
        g=m[m.role_final.eq(role)].copy()
        local_mask=mask[g.index]
        y=g.label_final.to_numpy(int)
        # fusion baseline cannot issue when AIA unavailable
        fusion_issue=~local_mask & np.isfinite(g.fusion_probability.to_numpy(float))
        # v3 routes detected AIA outage to SHARP fallback whenever SHARP exists
        v3_pred=g.v3_base_prediction.to_numpy(float).copy()
        v3_state=g.v3_base_state.astype(object).to_numpy()
        fallback=(local_mask & np.isfinite(g.sharp_probability.to_numpy(float)))
        v3_pred[fallback]=g.sharp_alarm.to_numpy(int)[fallback]
        v3_state[fallback]='DEGRADED'
        no_fallback=local_mask & ~np.isfinite(g.sharp_probability.to_numpy(float))
        v3_pred[no_fallback]=np.nan
        v3_state[no_fallback]='ABSTAIN'

        for system,issue,pred in [
            ('baseline_fusion',fusion_issue,g.fusion_alarm.to_numpy(float)),
            ('fallback_v3_q90',np.isfinite(v3_pred),v3_pred),
        ]:
            issued=np.asarray(issue,bool)
            met=binary_metrics(y[issued],np.asarray(pred[issued],int)) if issued.any() else {}
            rows.append({
                'scenario':scenario,'role':role,'system':system,'cases':len(g),
                'real_outage_cases':int(local_mask.sum()),
                'availability_rate':float(issued.mean()),
                'positive_retention_rate':float(issued[y==1].mean()) if (y==1).any() else np.nan,
                'abstained_flare_cases':int(((y==1)&~issued).sum()),
                **met
            })
    return rows

mask_incident=np.zeros(len(m),dtype=bool)
if len(mapped):
    mask_incident[mapped.row_index.astype(int).to_numpy()]=True
mask_persistent=np.zeros(len(m),dtype=bool)
if len(mapped):
    pm=mapped[mapped.persistent_after_recovery]
    if len(pm): mask_persistent[pm.row_index.astype(int).to_numpy()]=True

replay_rows=[]
replay_rows += replay(mask_incident,'all_observed_acquisition_incidents')
replay_rows += replay(mask_persistent,'persistent_after_recovery')
replay_df=pd.DataFrame(replay_rows)
replay_df.to_csv(OUT/'real_outage_replay_summary.csv',index=False)

# ------------------------------------------------------------
# Compare empirical incident structure with prior synthetic assumptions
# ------------------------------------------------------------
structure={
    'incident_samples_total':int(len(inc)),
    'incident_samples_by_year':{str(k):int(v) for k,v in inc.groupby('year').size().to_dict().items()},
    'incident_blocks_total':int(inc.block_id.nunique()),
    'recovery_classes':{str(k):int(v) for k,v in inc.recovery_class.value_counts().to_dict().items()},
    'mapped_to_graybox_cases':int(len(mapped)),
    'mapped_by_method':{str(k):int(v) for k,v in mapped.mapping_method.value_counts().to_dict().items()} if len(mapped) else {},
    'persistent_after_recovery_incident_ids':int(inc.persistent_after_recovery.sum()),
    'mapped_persistent_graybox_cases':int(mask_persistent.sum()),
    'block_span_hours_median':float(block_summary.span_hours.median()),
    'block_span_hours_max':float(block_summary.span_hours.max()),
}
(OUT/'empirical_outage_structure.json').write_text(json.dumps(structure,indent=2)+'\n')

protocol={
    'status':'POSTHOC_REAL_AIA_OUTAGE_REPLAY_COMPLETE',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'horizon_hours':72,
    'new_model_fitting':False,'new_threshold_tuning':False,'new_policy_selection':False,'new_policy_redesign':False,
    'outage_source':'actual upstream AIA acquisition/recovery inventories for 2025 and 2026',
    'outage_semantics':'detected acquisition failure / recovery incident; not silent corruption and not historical operational delivery evidence',
    'replay_scenarios':['all_observed_acquisition_incidents','persistent_after_recovery'],
    'mapping_policy':'exact source_sample_id when available, otherwise unique HARPNUM+issue-time match',
    'evidence_boundary':{'cycle25':'post-hoc robustness','2026':'post-hoc after locked evaluation; no redesign'},
    'source_sha256':{'master_predictions':MASTER_SHA,'sharp_predictions':SHARP_SHA,'cases':sha256(CASES)},
    'upstream_sources':source_manifest,
    'outputs':['upstream_source_manifest.csv','real_acquisition_incidents.csv','real_incident_summary.csv',
               'real_incident_blocks.csv','recovery_persistence.csv','incident_to_graybox_mapping.csv',
               'real_outage_replay_summary.csv','empirical_outage_structure.json'],
}
(OUT/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')

print('===== REAL AIA OUTAGE REPLAY COMPLETE =====')
print('Upstream AIA source:', AIA_REPO if AIA_REPO is not None else f'pinned GitHub commit {UPSTREAM_COMMIT}')
print('\nIncident summary:')
print(incident_summary.to_string(index=False))
print('\nEmpirical structure:')
print(json.dumps(structure,indent=2))
print('\nReplay summary:')
print(replay_df.round(6).to_string(index=False))
print('\nOutputs:',OUT)


## Interpretation guardrails

- Recovery inventories are **engineering acquisition incidents**, not proof that the same failures would have occurred in a live forecast service.
- `all_observed_acquisition_incidents` is a conservative replay of detected AIA failures before recovery; `persistent_after_recovery` represents the subset still missing in retained recovery evidence.
- The replay changes only AIA availability. It does not simulate silent corrupted pixels, wrong crops that passed validation, SHARP outages, latency, or simultaneous source failures.
- The frozen v3 policy is not retuned from these incidents.
